In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download  \n   - **macOS**: download and install the `.pkg`\n   - **Windows**: download and install the `.msi`\n   - **Linux**: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and start a model:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model, starts it locally, and opens a chat-like interface.\n\n3. To test that the local server is running, use:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'I can’t find any information in the FAQ context about running **Olama** locally.\n\nThe closest related note is that you **can run the course locally instead of Codespaces** if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module, and you should document your setup to keep it reproducible.\n\nIf you meant **Ollama**, that specific setup isn’t covered in the provided context.'

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can join if enrollment is still open.\n\nWhat to check:\n- **Course start date**: If it hasn’t started yet, you can likely enroll normally.\n- **Enrollment deadline**: Some courses close enrollment after a certain date.\n- **Seat availability**: If it’s capped, there may be a waitlist.\n- **Prerequisites**: Make sure you meet any requirements.\n\nIf you want, I can help you figure out the best way to ask the instructor or course admin.'

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course can I join it"}', call_id='call_PETsVyix8DUkyXbZL3Lu8UV0', name='search', type='function_call', id='fc_01ff1c179ced3925006ab28723b91c87d28dc45a07d88bd02f', async_=None, caller=None, namespace=None, status='completed')]

In [11]:
len(response.output)

1

In [12]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [15]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [16]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join. The course material is available, and you can start learning anytime.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions.'

In [17]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(786, 46)

In [18]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# Agentic Loop

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [23]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [24]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered just now enrollment access can I join"}
function_call: search {"query":"course enrollment new student join discovered course FAQ"}
function_call: search {"query":"can I join the course after it started FAQ"}


In [25]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while the course is still accepting submissions.

If you want, I can also help with questions about registration, homework deadlines, or certificates.


In [26]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [27]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install run locally Ollama"}
function_call: search {"query":"Ollama local setup run model localhost FAQ"}
function_call: search {"query":"run Ollama locally course FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - macOS: download and install the `.pkg` from https://ollama.com/download
   - Windows: download and install the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model if needed and opens an interactive chat in your terminal.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response indicating Ollama is up.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"

'To run Ollama locally:\n\n1. **Install Ollama**\n   - macOS: download and install the `.pkg` from https://ollama.com/download\n   - Windows: download and install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model if needed and opens an interactive chat in your terminal.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response indicating Ollama is up.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a “connection refused” error, restart the server with:\n```bash\nollama serve\n```\nor, in a notebook:\n```bash\

In [28]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered can I still join enrollment access FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.

If you'd like, I can also help with other course-related questions such as homework, certificates, or deadlines.


"Yes — you can still join the course.\n\nIf you want to receive a certificate, make sure you submit your project while submissions are still being accepted.\n\nIf you'd like, I can also help with other course-related questions such as homework, certificates, or deadlines."

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join late enrollment registration access FAQ"}
iteration #2...
function_call: search {"query":"certificate project submitting while accepting submissions live cohort self-paced can I still join FAQ llm zoomcamp"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If your goal is just to learn, you can start anytime. If you want a certificate, you need to submit your project while the course is still accepting submissions.

If you’d like, I can also explain what’s needed for the certificate or how the project submission works.


'Yes — you can still join the course.\n\nIf your goal is just to learn, you can start anytime. If you want a certificate, you need to submit your project while the course is still accepting submissions.\n\nIf you’d like, I can also explain what’s needed for the certificate or how the project submission works.'

In [30]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess queen's gambit opening definition"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening pawn c4 d5 definition"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. `1. d4 d5`
2. `2. c4`

White offers the c-pawn to try to gain control of the center and get a more active position. It’s called a “gambit” because one side offers material in the opening for positional advantage.

There are two common main types:
- **Queen’s Gambit Accepted**: Black takes the pawn on c4
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also explain:
- the main ideas behind the opening,
- how to play it as White or Black,
- or the difference between Queen’s Gambit and King’s Gambit.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. `1. d4 d5`\n2. `2. c4`\n\nWhite offers the c-pawn to try to gain control of the center and get a more active position. It’s called a “gambit” because one side offers material in the opening for positional advantage.\n\nThere are two common main types:\n- **Queen’s Gambit Accepted**: Black takes the pawn on c4\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also explain:\n- the main ideas behind the opening,\n- how to play it as White or Black,\n- or the difference between Queen’s Gambit and King’s Gambit.'

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening Queen's Gambit"}
iteration #2...
function_call: search {"query":"queen gambit course faq"}
iteration #3...
ASSISTANT:
I couldn’t find any course-related FAQ entry for “queen gambit,” so this looks off-topic for the course.

If you meant something else related to the course, please clarify. Otherwise, let me know if there are other areas you want to explore.


'I couldn’t find any course-related FAQ entry for “queen gambit,” so this looks off-topic for the course.\n\nIf you meant something else related to the course, please clarify. Otherwise, let me know if there are other areas you want to explore.'